<a href="https://colab.research.google.com/github/katyaangelimir-png/Machine-Learning/blob/main/Tarea2ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tarea 2 - Corina Tapia, Katya Mir


### Target: Variable binaria is_legendary, 1 si el Pokemon es legendario y 0 si es ordinario.
### Features: Las seis estadísticas de combate básicas como hp, attack, defense, sp_attack, sp_defense, speed y la variable de crianza base_egg_steps.
### Tipo del problema: Es un problema de clasificación binaria, ya que el objetivo es categorizar a cada Pokemon en una de dos clases mutuamente excluyentes, si es legendario u ordinario.
### Información útil: En el EDA se demostró que en la matriz de dispersión y las distribuciones de densidad los Pokémon legendarios ocupan una región distintiva en la periferia superior del espacio de características y poseen un presupuesto de poder total y un esfuerzo de crianza marcadamente superiores a los de la población general.

# Algoritmo de Machine Learning: Árbol de decisión

### Cumple con el objetivo principal de comprender cómo funciona el modelo y reconocer sus reglas. Un árbol de decisión permite visualizar de forma transparente los cortes exactos que determinan si un Pokémon es clasificado como legendario u ordinario.
### Como vimos en las figuras del análisis exploratorio, los legendarios no se separan mediante una línea recta simple, sino ocupando regiones específicas. El árbol puede dividir el espacio de características de manera ortogonal adaptándose a estos patrones.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score,)

In [ ]:
df = pd.read_csv('pokemon.csv')
df.head()
features = ['hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed', 'base_egg_steps',]
X = df[features]
y = df['is_legendary']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
modelo_arbol = DecisionTreeClassifier(random_state=42)
modelo_arbol.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [ ]:
y_pred_train = modelo_arbol.predict(X_train)
y_pred_test = modelo_arbol.predict(X_test)

print("Accuracy Train:", accuracy_score(y_train, y_pred_train))
print("Accuracy Test:", accuracy_score(y_test, y_pred_test))

print(classification_report(y_test, y_pred_test, target_names=["Ordinario", "Legendario"]))

print("Matriz de Confusion Test:")
print(confusion_matrix(y_test, y_pred_test))

Accuracy Train: 1.0
Accuracy Test: 0.9751552795031055
              precision    recall  f1-score   support

   Ordinario       0.98      0.99      0.99       147
  Legendario       0.92      0.79      0.85        14

    accuracy                           0.98       161
   macro avg       0.95      0.89      0.92       161
weighted avg       0.97      0.98      0.97       161

Matriz de Confusion Test:
[[146   1]
 [  3  11]]


### Luego de evaluar las métricas, se eligió el F1-score como métrica principal porque es ideal para problemas con desbalance de clases como los Pokémon legendarios, que son muy pocos frente a los ordinarios. Al combinar la precisión y el recall en una sola medida, nos dice qué tan bueno es el modelo clasificando a la clase minoritaria sin dejarse sesgar por el volumen de la clase mayoritaria.
### Por otro lado, la matriz de confusión nos dice que de 14 legendarios reales en el conjunto de prueba, el modelo identificó correctamente a 11 y solo se equivocó en 3 falsos negativos y 1 falso positivo.

# Un experimento con el modelo

### Como vimos en la evaluación anterior, el modelo original no tenía límite de profundidad, lo que produjo un Accuracy Train de 1.0, es decir, una memorización del conjunto de entrenamiento. Al restringir la profundidad máxima del árbol de decisión, analizaremos cómo cambia la presición.
### Predicción: Al limitar la profundidad máxima del Árbol de Decisión, se espera que la precisión en el conjunto de entrenamiento disminuya ligeramente y ya no sea 1.0, ya que el árbol no podrá memorizar casos especiales u outliers.

In [ ]:
modelo_arbol_exp = DecisionTreeClassifier(max_depth=3, random_state=42)
modelo_arbol_exp.fit(X_train, y_train)
y_pred_train_exp = modelo_arbol_exp.predict(X_train)
y_pred_test_exp = modelo_arbol_exp.predict(X_test)

In [ ]:
resultados = { 'Modelo': ['Original', 'Modificado'],'Accuracy Train': [accuracy_score(y_train, y_pred_train), accuracy_score(y_train, y_pred_train_exp),]
              ,'Accuracy Test': [accuracy_score(y_test, y_pred_test), accuracy_score(y_test, y_pred_test_exp),]
              ,'F1-Score Test': [f1_score(y_test, y_pred_test, pos_label=1), f1_score(y_test, y_pred_test_exp, pos_label=1),],}

df_resultados = pd.DataFrame(resultados)
print(df_resultados)

       Modelo  Accuracy Train  Accuracy Test  F1-Score Test
0    Original        1.000000       0.975155       0.846154
1  Modificado        0.990625       0.981366       0.880000


### Interpretación: Se cumplió la predicción, ya que al restringir la profundidad máxima del árbol, se logró evitar el sobreajuste en el entrenamiento. Al no permitir que el modelo crezca de forma descontrolada, este descarta reglas específicas y aprende patrones generales de mejor manera. Este experimento demuestra que un modelo más simple y restringido no solo previene el sobreajuste frente a los datos de entrenamiento, sino que mejora su capacidad de generalización en datos nuevos.

# Auditoría del modelo

### 1. ¿Qué tan bien generaliza el modelo?
###El modelo original presenta un Accuracy de 1.0 en el conjunto de entrenamiento, pero baja a un 0.975 en el conjunto de prueba, lo que indica claros indicios de sobreajuste debido a que el árbol creció sin restricciones y memorizó detalles específicos o ruido de los datos de entrenamiento. Sin embargo, al restringir la profundidad, el modelo mejoró su generalización, el Accuracy en test subió a 0.981 y el F1-Score aumentó de 0.846 a 0.880, demostrando que un modelo regularizado generaliza mejor ante datos no vistos.

###2. ¿Dónde falla el modelo?
###Al revisar las métricas de evaluación del conjunto de prueba notamos que el modelo falló en dos situaciones: al reconocer 3 falsos negativos y al reconocer 1 falso positivo.
###Los falsos negativos se deben a Pokémon legendarios de estadísticas base más bajas que se solapan en el espacio de los Pokémon comunes más fuertes.
###Los falsos positivos se deben a Pokemón comunes con stats muy altas que confunden al árbol categorizandolos como Legendarios.

###3. ¿Qué aprendieron que no era evidente a partir de una única métrica global?
###Que el Accuracy oculta el verdadero comportamiento del modelo frente al desbalance de clases, ya que si solo vemos esta métrica podríamos pensar que el modelo es perfecto, pero al utilizar la matriz de confusión y el F1-score descubrimos que le cuesta más capturar a la clase minoritaria (los legendarios). Por otra parte, el experimento demostró que penalizar la complejidad del árbol no empeora el rendimiento, sino que soluciona el sobreajuste y mejora la detección de clases.

###4. ¿Cómo mejorarían el modelo?
###El modelo podría mejorar usando un Random Forest, ya que combinan múltiples árboles de decisión para reducir la varianza, disminuir el sobreajuste y manejar de mejor forma las regiones donde se confunden los Pokémon comunes de alto poder con los legendarios.

###Conclusiones
###Considerando las métricas obtenidas para los conjuntos de prueba, nuestro modelo tiene una confiabilidad bastante alta, a pesar de que no sea perfecto.
###Una fortaleza de nuestro modelo, es que es capaz de identificar con gran precisión la mayoría de los Pokémon legendarios basándose principalmente en las estadísticas de combate y condiciones de crianza, entregandonos una fácil interpretación gracias a la estructura transparente del árbol de decisión.
### Una limitación de nuestro modelo es su vulnerabilidad en la frontera de clasificación frente a Pokémon no legendarios extremadamente poderosos (falsos positivos) o legendarios de bajo perfil (falsos negativos). Esto demuestra que una métrica global alta no garantiza que el modelo no cometa errores importantes en casos límites, por lo que su aplicación en un entorno real requeriría precauciones adicionales.